In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv 


In [2]:
load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [4]:

from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages: str


In [5]:
def chat_node(state: ChatState) -> ChatState:
    messages = state.get('messages', [])
    response = model(messages)
    messages.append(response)
    return {'messages': messages}

In [6]:
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile()

In [ ]:
while True:
    user_input = input("User: ")
    print("User: ",user_input)
    if user_input.strip().lower() in ['exit', 'quit','bye']:
        break
    response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]})
    print("Bot:", response['messages'][-1].content)
